In [1]:
from pathlib import Path
import numpy as np
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from spike_classifier.annotate_spikes import annotate_spikes
from spike_classifier.train_classifier import train_spike_classifier
from spike_classifier.prepare_data import prepare_spike_data
from utils.label_utils import reset_spike_labels


In [2]:

ROI_DATA_PATH = PROJECT_ROOT / "data" / "invivo_roi_features.npy"
assert ROI_DATA_PATH.exists(), f"Data file not found: {ROI_DATA_PATH}"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config" / "classifier_config.yaml"

print(f"Path to data: {ROI_DATA_PATH}")
print(f"Saving models to: {MODEL_OUT_DIR}")
print(f"Config: {CONFIG_PATH}")

Path to data: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy
Saving models to: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:

reset = False # Gated purposely to avoid accidental resets
if reset:
    roi_dict = np.load(ROI_DATA_PATH, allow_pickle=True).item()
    roi_dict, n_reset = reset_spike_labels(roi_dict)
    np.save(ROI_DATA_PATH, roi_dict, allow_pickle=True)
    print(f"Reset {n_reset} labels to unlabeled")

In [3]:
roi_dict = prepare_spike_data(
    input_path=str(ROI_DATA_PATH),
    output_path=None,  
    max_rois=None,
    fs=3.0             # Frame rate in Hz — adjust to match your acquisition rate
)


Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy

  ROI Summary
  Total rois: 1942
  Good: 144 | Bad: 357 | Unlabeled: 1441
  Manual: 500 | Auto: 1
  Total spikes stored: 5301



In [4]:
n_samples = 5000
unlabeled_only = True
labeled_only = False
checkpoint_interval = 1000

annotate_spikes(
    data_path=ROI_DATA_PATH,
    max_rois=n_samples,
    unlabeled_only=unlabeled_only,
    labeled_only=labeled_only,
    checkpoint_interval=checkpoint_interval,
    verbose=True
)


Session ended by user. Saving progress...
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy

  SPIKE Annotation Summary
  Queued:    5265
  Seen:      646
  Labeled:   646
  Updated:   644
  Confirmed: 2
  Skipped:   0


  SPIKE Summary
  ROIs: 1942  (186 with spikes)
  Total spikes: 5301
  Good: 179 | Bad: 479 | Unlabeled: 4643
  Manual: 658 | Auto: 0



{'level': 'spike',
 'queued': 5265,
 'total': 646,
 'labeled': 646,
 'updated': 644,
 'confirmed': 2,
 'skipped': 0,
 'queued_rois': 186}

In [5]:

name = "invivo_spike_classifier" # TODO Change as desired for your organizational needs

results = train_spike_classifier(
    config_path=CONFIG_PATH,
    data_path=ROI_DATA_PATH,
    name=name,
    output_dir=MODEL_OUT_DIR,
    verbose=True,
    manual_only=True
)


Dataset Summary
--------------------------------------------------
Total labeled datapoints: 658
  Train: 526 | Test: 132

Label distribution:
              Bad (0)  Good (1)
  Train           378       148
  Test            101        31
  Total           479       179

Training on: Manual labels only

--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     LogisticRegression
Transform: sqrt
Features:  ['mini_prom', 'spike_prom', 'distance']

Hyperparameters:
  C: 1
  class_weight: None
  max_iter: 500
  penalty: l1
  solver: saga

Metrics:
  CV Accuracy:   0.9145
  Test Accuracy: 0.9242
  ROC AUC:       0.9658
  F1:            0.9224
  Precision:     0.9229
  Recall:        0.9242

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    98      3      
  Actual 1    7       24     
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models\inv